# Template 02: Data Conditioning

**Purpose:** Clean and condition assembled data

**Inputs:**
- data/01_assembled.parquet (from Template 01)

**Outputs:**
- data/02_conditioned.parquet

**Processing:**
- Set proper data types
- Handle missing values
- Basic feature validation

In [ ]:
# Parameters (injected by papermill)
config_path = "config/car_coll/v1"

In [ ]:
# Imports
import pandas as pd
import numpy as np
import yaml
import os
import sys
import gc
from pathlib import Path

# Setup environment
sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment

print("########################################")
print("# STAGE 02: DATA CONDITIONING")
print("########################################")

project_root = setup_notebook_environment()
print(f"\nProject root: {project_root}")

In [ ]:
# Load config
config_file = f"{config_path}/config.yaml"
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

output_base = cfg['paths']['output_base']
print(f"\n* Experiment: {cfg['experiment']['name']}")
print(f"* Target: {cfg['experiment']['target']}")

In [ ]:
# Load checkpoint from Template 01
checkpoint_01 = f"{output_base}/data/01_assembled.parquet"
print(f"\n* Loading checkpoint: {checkpoint_01}")

data = pd.read_parquet(checkpoint_01)
print(f"  Shape: {data.shape}")
print(f"  Memory: {data.memory_usage(deep=True).sum() / 1e9:.2f} GB")

In [ ]:
# Check data types
print(f"\n* Data type distribution:")
print(data.dtypes.value_counts())

In [ ]:
# Check for missing values
missing_counts = data.isnull().sum()
missing_cols = missing_counts[missing_counts > 0]

if len(missing_cols) > 0:
    print(f"\n* Missing values found in {len(missing_cols)} columns:")
    for col, count in missing_cols.head(10).items():
        pct = (count / len(data)) * 100
        print(f"  {col}: {count:,} ({pct:.2f}%)")
    if len(missing_cols) > 10:
        print(f"  ... and {len(missing_cols) - 10} more")
else:
    print(f"\n* No missing values found")

In [ ]:
# Apply type conversions from config
type_conv_file = f"{config_path}/type_conversions.csv"

if os.path.exists(type_conv_file):
    print(f"\n* Applying type conversions from {type_conv_file}")
    conversions = pd.read_csv(type_conv_file)
    for _, row in conversions.iterrows():
        col = row['column_name']
        if col in data.columns and row['target_dtype'] == 'numeric':
            data[col] = pd.to_numeric(data[col], errors='coerce')
            print(f"  ✓ {col}: object → numeric")
else:
    print(f"\n* No type conversions file found, skipping")

# Basic conditioning: Fill missing values with 0 for numeric features
# (More sophisticated strategies can be added later)
print(f"\n* Conditioning data...")

# Get numeric columns
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
print(f"  Numeric columns: {len(numeric_cols)}")

# Fill missing values in numeric columns
for col in numeric_cols:
    if data[col].isnull().any():
        data[col] = data[col].fillna(0)

print(f"  Missing values filled with 0")

In [ ]:
# Verify target variable exists
target = cfg['experiment']['target']
if target in data.columns:
    print(f"\n* Target variable '{target}' verified")
    print(f"  Non-null count: {data[target].notnull().sum():,}")
    print(f"  Mean: {data[target].mean():.4f}")
    print(f"  Median: {data[target].median():.4f}")
    print(f"  Min: {data[target].min():.4f}")
    print(f"  Max: {data[target].max():.4f}")
else:
    print(f"\n* ERROR: Target variable '{target}' not found!")
    print(f"  Available columns: {data.columns.tolist()[:20]}")

In [ ]:
# Save conditioned checkpoint
checkpoint_02 = f"{output_base}/data/02_conditioned.parquet"
print(f"\n* Saving checkpoint: {checkpoint_02}")

data.to_parquet(checkpoint_02)
file_size = os.path.getsize(checkpoint_02) / 1e9

print(f"  Saved: {file_size:.2f} GB")
print(f"  Shape: {data.shape}")

In [ ]:
# Memory cleanup
print(f"\n* Cleaning memory...")
del data
gc.collect()
print(f"  Memory cleaned")

In [ ]:
print("\n########################################")
print("# STAGE 02: COMPLETE")
print("########################################")